In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay,
)


In [ ]:
# ======================
# 1. ПАРАМЕТРЫ
# ======================
DATA = Path("../data/churn.csv")
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 2


In [ ]:
data = pd.read_csv(DATA)


In [ ]:
print("Number of rows:", data.shape[0])
print("churn rate:", round(data["churn"].mean(), 3))
data.head()


In [ ]:
data.info()


In [ ]:
# Баланс классов
plt.figure(figsize=(6, 4))
sns.countplot(data=data, x='churn', palette='rocket', hue='churn', legend=False)
plt.title('Баланс churn', fontsize=13)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# ======================
# 2. ПРИЗНАКИ И МОДЕЛЬ
# ======================
num_cols = ["tenure_months", "monthly_charges", "support_tickets", "total_charges"]
cat_cols = ["contract_type", "internet_service"]

X = data[num_cols + cat_cols]
y = data["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_SEED
)


In [ ]:
pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])),
    ("clf", RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_SEED)),
])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
proba = pipe.predict_proba(X_test)[:, 1]


In [ ]:
# ======================
# 3. МЕТРИКИ ОДНОЙ ТАБЛИЦЕЙ
# ======================
metrics = {
    "accuracy": accuracy_score(y_test, pred),
    "precision": precision_score(y_test, pred),
    "recall": recall_score(y_test, pred),
    "f1": f1_score(y_test, pred),
    "roc_auc": roc_auc_score(y_test, proba),
}
metrics_df = pd.DataFrame([metrics]).T.rename(columns={0: "value"}).round(3)
metrics_df


In [ ]:
# Столбчатая диаграмма метрик
plt.figure(figsize=(8, 5))
sns.barplot(
    x=metrics_df.index,
    y=metrics_df["value"],
    hue=metrics_df.index,
    palette="viridis",
    legend=False,
)
plt.title("Classification metrics cheat sheet", fontsize=14, pad=15)
plt.ylabel("Значение", fontsize=12)
plt.ylim(0, 1.05)
plt.grid(axis="y", linestyle="--", alpha=0.5)

for i, v in enumerate(metrics_df["value"]):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig(FIG / "metrics_bar.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ======================
# 4. CONFUSION MATRIX
# ======================
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, colorbar=False)
ax.set_title("Confusion matrix")
plt.tight_layout()
plt.savefig(FIG / "metrics_confusion.png", dpi=120, bbox_inches="tight")
plt.show()

cm = confusion_matrix(y_test, pred)
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")


In [ ]:
# ======================
# 5. ROC И PR-КРИВЫЕ
# ======================
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

RocCurveDisplay.from_predictions(y_test, proba, ax=axes[0], name="RF")
axes[0].set_title("ROC-кривая")
axes[0].grid(True, linestyle="--", alpha=0.5)

PrecisionRecallDisplay.from_predictions(y_test, proba, ax=axes[1], name="RF")
axes[1].set_title("Precision-Recall")
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(FIG / "metrics_roc_pr.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ======================
# 6. ПОРОГ: precision/recall trade-off
# ======================
thresholds = np.linspace(0.1, 0.9, 17)
rows = []
for thr in thresholds:
    p = (proba >= thr).astype(int)
    rows.append({
        "threshold": thr,
        "precision": precision_score(y_test, p, zero_division=0),
        "recall": recall_score(y_test, p, zero_division=0),
        "f1": f1_score(y_test, p, zero_division=0),
    })
thr_df = pd.DataFrame(rows)

plt.figure(figsize=(9, 5))
plt.plot(thr_df["threshold"], thr_df["precision"], label="precision", marker="o", ms=4)
plt.plot(thr_df["threshold"], thr_df["recall"], label="recall", marker="o", ms=4)
plt.plot(thr_df["threshold"], thr_df["f1"], label="f1", marker="o", ms=4)
plt.title("Метрики vs порог вероятности", fontsize=14, pad=15)
plt.xlabel("Порог", fontsize=12)
plt.ylabel("Значение", fontsize=12)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIG / "metrics_threshold.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
print(metrics_df.to_string())
print('TN/FP/FN/TP зависят от порога 0.5 по умолчанию.')


In [ ]:
print("AUC — для ранжирования; F1/precision/recall — для рабочего порога.")
print("Accuracy одна — плохой выбор при дисбалансе.")
